## Stvarni čas (Real-time)
- Je cas, ki ga mi zaznavamo kot realnega.  
Primer: Če A nekaj snama in nam pošilja, mi pa na B napravi to gledamo, če se nam zdi da gledamo v živo je v živo.
- Tok podatkov mora biti konstanten
- Primeri:
    - Enosmerni tok: TV, predstavitve, zvok
    - Dvosmerni: Video klic, video telefonija

- Deluje na principu, da A posname nek dogodek, in ga pošlje
- B pa ga prejme (podatke, cas posnetka paketa, št. paketa)
- Celotna slika pase gradi na aplikacijski plasti

### Težave:
- Omrežje
    - Vsak paket lahko gre po drugi poti
    - Vsak potuje različno dolgo
    - Nekateri paketi se lahko izgubijo
- Kaj narediti z izgubljenimi paketi:
    - **Pakete obravnavamo kot izgubljene. To naredi aplikaciska plast** (UDP)
- Kaj narediti z neenakomirno prihajajočimi paketi
    - **Aplikaciska plast jih sortira. Če ima paket podvojen čas, gledamo indexse (preprečimo podajanje paketov)**
    - **Če paket pride preporno ga razglasimo kot izgubljenega**

## RTP - Real time Protocol
[RTP_packet](./Slike/RTP_packet.png)    (V, P, sequence number, timestamp)  
[RTP_packet1](./Slike/RTP_packet1.png)  (SSRC, CC, CSRC)    
[RTP_packet2](./Slike/RTP_packet2.png)  (PT, M, X)  
**Data: čas narejene slike, index paketa, tok** 
Poskrbi, da čim več podatkov pride na drugo stran
1. skrbi za pravo zaporedje paketov
2. skrbi za časovne značke dogodkov
3. Nimamo začetka ali konca seje (UDP) - **Drug protokol pomaga pri temu (RTSP)**
- Zna za vsak tip podatkov, ki jih pošilja vzpostaviti svoj tok - **Ima več tokov**
- Ima poseben element – mešalec (mixer), ki lahko združuje več sej v eno sejo

**Deli paketa:**
- V = verzija
- P = zapolnitev (padding)
- sequence number = št. pateta poslanega v toku
- timestamp = časovna značka dogodke (kdaj je slika, zvok nastal)
- **SSRC = indetifikator vira (notranjega toka)**
- **CC = št. mešanih virov (tokov) v enem toku**
- **CSRC = indetifikatorji mešanih virov (glavnega toka)**
- PT = indetifikator protokola
- M = posebni bit za potrebe protokola
- X = posebni bit za razširitev glave


## RTCP - Real Time Control Panel
[RTCP_packet](./Slike/RTCP_packet.png)
1. Kakovost prenesenega prometa (večja frekvenca zvoka, več pikslov na sliki)
2. dodaten opis vira tako dogodkov
3. poskrbi zapravilno gostoto pošiljanja
4. pošilja lahko še dodatne informavije na zahtevo aplikacije

- Stalna pasovnaširina
- Če je veliko sodelujočih strank potem je gostota manjša

**CNAME omogoča:**
- Prepoznavo istega udeleženca, tudi če se njegov SSRC spremeni
- Sinhronizacijo medija iz različnih tokov istega uporabnika

**NAME** 
- Prikazije ime naprave, lokacije, uporabnika.


## RTP Varnost
- Uporablja UDP brez varnosti (nima SSL)    
[SSL/TLS](./Slike/SSL/TLS.png)      
SSL1.0 -> SSL2.0 -> SSL3.0 -> TLS1.0 -> TLS1.1 -> TLS1.2 -> **TLS1.3**      

0. **SRTP + DTLS**
- SRTP omogova avtentifikacijo, kriptavijo, celovitost, urejajo ponavljanje
- DTLS je TLS/SSL z TCP, ki je namenjen za uporabo z UDP protokolom. Zagotovi nam prenos kljucev za SRTP.

1. **RFC + SSL** oz. UDP + SSL (določiti nam:)
- začetek
- prenašanje
- zakljucek

2. **SRTP**: - The Secure Real-time Transport Protocol  
[Tokovna_šifra](./Slike/Tokovna_sifra.png)  
- Varni protokol RTP (RTP+tokovna šifra)
- Dodana tokovna šifra
- Moramo imeti:
    - začetna vrednost (IV) je poznana obema stranema
    - obema stranema je poznan tudi ključ
    - vsak paket se ločeno zakriptira
    - je preprosti xor ali kakšen podoben algoritem
    - če se paket izgubi, samo v prazno zavrtimo E

- Programska oprema:
    -  eden prvih odprtokodnih strežnikov je **Darwin**

## RTSP- Real time streaming protocol
- Omogoča nam izdelavo začetka in konca protokola
- Deluje na principu video rekorderja (**start, stop, rewined**)
- **Uporabljamo za začetek komuniciranja**  
✔ reče strežniku kateri medij naj pripravi     
✔ določi način prenosa (RTP/UDP, TCP, multicast …)     
✔ začne predvajanje (PLAY)     
✔ ga zaustavi (PAUSE)  
✔ ga konča (TEARDOWN)  
✔ lahko premika naprej/nazaj (SEEK)    

**RTSP torej upravlja življenjski cikel streama, ne pa vsebine.**   


## NTP - Network time protocol
Konfiguracija na FreeBSD: [/etc/ntp.conf]   
- [NTP_packet](./Slike/NTP_packet.png)
- [Packet_cnfiguration](./Ostalo_Ideje.ipynb) (Mode, VN,...)
- Omogoča, da imajo vsi sistemi v omrežju skoraj enak čas, običajno z natančnostjo milisekund ali še boljšo.
- Čas se meri na satelitih (atomske ure)
- Nato pa ga pošljemo na korenske strežnike za NTP. Ti se delijo na manjše, ...
Pro protokolu moramo upoštevati: (**Na podlagi tega računamo zamik ure**)(upoštevamo pot, ki je pripotovala do nas)
- T1 – čas, ko odjemalec pošlje zahtevo
- T2 – čas, ko strežnik prejme zahtevo
- T3 – čas, ko strežnik pošlje odgovor
- T4 – čas, ko odjemalec prejme odgovor

- t = pošiljanje in prejemanje na uporabniku in NAT strežniku
- δ = (t4-t1) - (t3-t2) (Omrežna zakasnitev v obe smeri)
- NAT čas = **t4 + ( (t2 - t1) + (t3 - t4) ) / 2**    

*Ure ne smemo predtavljate (zaradi procesov na računalniku, ki se morda odvisni od časa).*
- Zato jo lahko samo pohitrimo ali upočasnimo glede na razliko med dejansko in normalno uro

# EFI in UTC

### UTC (Coordinated Universal Time)
- UTC je **koordiniran univerzalni čas**, mednarodni standard.
- Osnova za vse **časovne pasove**.
- **Ne spreminja se** zaradi poletnega časa.
- Uporablja se za:
  - log datoteke,
  - sinhronizacijo strežnikov,
  - omrežne protokole,
  - baze podatkov.
- Primer:
  - UTC = 12:00
  - Slovenija (CET, +1) = 13:00
  - Poletni čas (CEST, +2) = 14:00

### EFI / RTC ura
- EFI (Extensible Firmware Interface) ali starejši BIOS vsebuje **strojno uro** (RTC).
- Ura deluje **neodvisno od OS** in hrani čas, ko je računalnik izklopljen.
- Uporablja se za:
  - določitev časa ob zagonu OS,
  - inicializacijo sistemskega časa.
- EFI ura se lahko nastavi kot **UTC** ali **lokalni čas**.
- Problem:
  - Windows privzeto misli, da je EFI ura **lokalni čas**.
  - Linux privzeto misli, da je EFI ura **UTC**.
  - Posledica: napačen prikaz časa (npr. ena ura premalo ali preveč).


### Zvok
- Je analogni signal    
Za predvajanje ga moramo zajeti digitalni, v diskretnih trenutkih. (Ne smemo izgubiti preveč informaci) 
Zajemamo odmik:
    - amplitudo,
    - jakost,
    - energijo
- Amplitudo predstavimo z n-bitno št.
- Predstavljen pa je kot linearna kombinacija več sinusnih signalov.
- Problemi:
    - problem vzorčenja (Nyquist-ova frekvenca) (kolesa se vrtijo nazaj)
        - ne moramo narediti dovolj slik (previsoka frekvenca vrtenja koles > 12Hz) (zdi se da se vrtijo nazaj)
    - človeško uho zaznava frekvence približno od 20Hz do 22kHz -> (frekvenca wav 44.1kHz)
    - človeško uho ne zazna določene kombinacije signalov (mp3 stiskanje)


## Slika
Sliko predstavimo:
- vsaka točka na zaslonu ima svojo vrednost, ki je tri razsežnostni vektor
    - Prostorske: tridimenzionalne -> (x, y, z) -> določajo pozicijo ali premik v prostoru.
    - Vektor hitrosti: vx – hitrost v smeri x, vy – hitrost v smeri y, vz – hitrost v smeri z
    - Vektor slike: Fx, Fy, Fz
    - Barni vektor: RGB
- problem časovne digitalizacije je podoben / enak kot pri zvoku
    - človeško oko zazna neprekinjeno premikanje, če mu posredujemo vsaj med 23 do 25 slikic na sekundo